# GW170817 PE — Fixed sky, reduced frequency grid

Parameter estimation of GW170817 with **sky location fixed** to NGC 4993,
using the **mlgw_bns_jax** waveform model and **SHARPy** SMC sampler.

This notebook uses a **reduced frequency grid** (~4000 points instead of ~253k)
to drastically cut GPU memory, allowing many more SMC particles on a single A100.

The technique:
1. Build the GW network at full FFT resolution (df = 1/128 Hz).
2. Phase-rotate the data by `exp(+2πi f (T−1))` to centre around trigger time.
3. Resample data and PSD onto a coarse uniform grid.
4. Replace the detector arrays via `batched_detector.replace()`.
5. Monkey-patch `project_waveform` to remove the `(T−1)` phase (already absorbed).

## Environment setup (Colab / fresh environment)

This cell installs all required packages and clones the repositories.
**Skip on a local machine** where everything is already installed.

In [ ]:
import os, subprocess, sys

COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

if COLAB:
    # ── Install JAX with CUDA 12 support ─────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    # ── Install other Python packages ────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "corner", "gwpy", "h5py", "ripplegw", "lalsuite", "jaxopt", "netket",
    ])
    # ── Install BlackJAX fork (SHARPy needs custom build_kernel) ─────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps",
        "blackjax @ git+https://github.com/gabrieledemasi/blackjax@main",
    ])

    # ── Clone the main repo (contains model, waveform loader, etc.) ──
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "jax_mlgw_bns", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git",
            REPO_DIR,
        ])

    # ── Clone SHARPy ─────────────────────────────────────────────────
    sharpy_repo = os.path.join(REPO_DIR, "_sharpy_repo")
    sharpy_pkg  = os.path.join(sharpy_repo, "sharpy")
    sharpy_link = os.path.join(REPO_DIR, "sharpy")
    if not os.path.isdir(sharpy_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/sharpy.git",
            sharpy_repo,
        ])
    if not os.path.exists(sharpy_link):
        os.symlink(sharpy_pkg, sharpy_link)

    # ── Fix blackjax circular import (Python 3.11 compatibility) ─────
    import site
    for _sp in site.getsitepackages():
        _chees = os.path.join(_sp, "blackjax", "adaptation", "chees_adaptation.py")
        if os.path.exists(_chees):
            with open(_chees, "r") as f:
                _csrc = f.read()
            _old_ci = "import blackjax.optimizers.dual_averaging as dual_averaging"
            _new_ci = "from blackjax.optimizers import dual_averaging"
            if _old_ci in _csrc:
                _csrc = _csrc.replace(_old_ci, _new_ci)
                with open(_chees, "w") as f:
                    f.write(_csrc)
                print("Patched chees_adaptation.py: fixed circular import.")
            break

    # ── Fix ripplegw import compatibility in SHARPy ──────────────────
    _gw_lik = os.path.join(sharpy_pkg, "GW_likelihood.py")
    _old_import = "from ripplegw import ms_to_Mc_eta"
    with open(_gw_lik, "r") as f:
        _src = f.read()
    if _old_import in _src and "try:" not in _src.split(_old_import)[0][-30:]:
        _new_import = (
            "try:\n"
            "    from ripplegw import ms_to_Mc_eta\n"
            "except ImportError:\n"
            "    def ms_to_Mc_eta(m):\n"
            "        m1, m2 = m\n"
            "        return (m1 * m2) ** (3 / 5) / (m1 + m2) ** (1 / 5), m1 * m2 / (m1 + m2) ** 2"
        )
        _src = _src.replace(_old_import, _new_import)
        with open(_gw_lik, "w") as f:
            f.write(_src)
        print("Patched GW_likelihood.py: ms_to_Mc_eta import made robust.")

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
    print(f"SHARPy: {sharpy_link} -> {sharpy_pkg}")
else:
    print("Not running on Colab — skipping setup.")

## Download GWOSC data (BayesWave-cleaned L1)

Downloads 1024 s of 4 kHz strain from GWOSC for H1, L1 and V1.

For L1, the scatter-light glitch near the merger is subtracted using the
BayesWave-cleaned frame from DCC LIGO-T1700406-v3 (spliced at GPS 1187008667).

In [ ]:
import os, sys, time, shutil
import numpy as np

_GPS_START = 1187008114
_DURATION  = 1024
_SRATE     = 4096
_DATA_DIR  = "gw170817_data"
os.makedirs(_DATA_DIR, exist_ok=True)

_DETECTORS = ["H1", "L1", "V1"]

# ── BayesWave-subtracted L1 data from DCC LIGO-T1700406-v3 ──────────
_DCC_GWF_URL = (
    "https://dcc.ligo.org/public/0144/T1700406/003/"
    "L-L1_CLEANED_HOFT_C02_T1700406_v3-1187008667-4096.gwf"
)
_DCC_CHANNEL = "L1:DCH-CLEAN_STRAIN_C02_T1700406_v3"
_DCC_GPS0    = 1187008667
_DCC_SRATE   = 16384


def _ensure_gwf_backend():
    """Make sure at least one GWF reader is importable."""
    for mod in ("frameCPP", "lalframe", "framel"):
        try:
            __import__(mod)
            return
        except ImportError:
            pass
    conda = shutil.which("conda") or shutil.which("mamba")
    if conda:
        import subprocess
        for pkg in ("framel", "python-lalframe"):
            print(f"  Trying: {conda} install -c conda-forge {pkg}", flush=True)
            ret = subprocess.call([conda, "install", "-c", "conda-forge", "-y", "-q", pkg])
            if ret == 0:
                return
    import subprocess
    for pkg in ("framel",):
        ret = subprocess.call([sys.executable, "-m", "pip", "install", "-q", pkg])
        if ret == 0:
            return
    raise ImportError(
        "Cannot read GWF files. Install a backend manually:\n"
        "  conda install -c conda-forge framel\n"
    )


def _read_gwf_channel(path, channel, start, end):
    """Read a single channel from a GWF file (try gwpy then framel)."""
    try:
        from gwpy.timeseries import TimeSeries
        ts = TimeSeries.read(path, channel, start=start, end=end)
        return np.asarray(ts.value, dtype=np.float64), float(ts.sample_rate.value)
    except Exception:
        pass
    import framel
    vec = framel.frgetvect1d(path, channel, start, end - start, 0)
    data = np.asarray(vec[0], dtype=np.float64)
    sr   = 1.0 / vec[3]
    return data, sr


_all_exist = all(
    os.path.isfile(os.path.join(_DATA_DIR,
        f"{d[0]}-{d}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt"))
    for d in _DETECTORS
)

if _all_exist:
    print("Cleaned data files already exist — skipping download.")
else:
    from gwpy.timeseries import TimeSeries
    from scipy.signal import decimate as _decimate

    for det in _DETECTORS:
        out_file = os.path.join(_DATA_DIR,
            f"{det[0]}-{det}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt")

        if det == "L1":
            print("L1: building cleaned timeseries")
            t0 = time.time()

            print("  Downloading raw L1 from GWOSC...", flush=True)
            ts_raw = TimeSeries.fetch_open_data(
                "L1", _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)

            gwf_local = os.path.join(_DATA_DIR, "L1_cleaned_bw_T1700406.gwf")
            if not os.path.isfile(gwf_local):
                import requests
                print("  Downloading BayesWave GWF from DCC (~1 GB)…", flush=True)
                resp = requests.get(_DCC_GWF_URL, stream=True)
                resp.raise_for_status()
                with open(gwf_local, "wb") as fout:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        fout.write(chunk)
                print(f"  Saved GWF ({os.path.getsize(gwf_local)/1e6:.0f} MB)")
            else:
                print("  BayesWave GWF already cached.")

            _ensure_gwf_backend()

            _need_end = _GPS_START + _DURATION
            print("  Reading cleaned segment from GWF...", flush=True)
            bw_data, bw_sr = _read_gwf_channel(
                gwf_local, _DCC_CHANNEL, _DCC_GPS0, _need_end)

            if int(round(bw_sr)) != _SRATE:
                factor = int(round(bw_sr)) // _SRATE
                print(f"  Resampling {int(bw_sr)} -> {_SRATE} Hz (factor {factor})")
                bw_data = _decimate(bw_data, factor, ftype="iir", zero_phase=True)

            n_raw = int((_DCC_GPS0 - _GPS_START) * _SRATE)
            strain = np.concatenate([ts_raw.value[:n_raw], bw_data])
            n_expected = _DURATION * _SRATE
            assert len(strain) == n_expected, (
                f"L1 splice length mismatch: {len(strain)} vs {n_expected}")

            with open(out_file, "w") as fw:
                fw.write("# BayesWave-cleaned L1 strain for GW170817\n")
                fw.write(f"# GPS [{_GPS_START}, {_GPS_START+_DURATION}], "
                         f"splice at GPS {_DCC_GPS0}\n")
                fw.write("# Before splice: raw GWOSC.  After: DCC T1700406-v3 (BayesWave)\n")
                fw.write(f"# {_SRATE} samples per second\n")
                for val in strain:
                    fw.write(f"{val:.16e}\n")
            print(f"  -> L1 done in {time.time()-t0:.1f}s")

        else:
            print(f"{det}: downloading {_DURATION}s from GWOSC…", flush=True)
            t0 = time.time()
            ts = TimeSeries.fetch_open_data(
                det, _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)
            with open(out_file, "w") as fw:
                fw.write(f"# {det}: raw GWOSC strain for GW170817\n")
                fw.write(f"# {_SRATE} samples per second\n")
                fw.write(f"# starting GPS {_GPS_START} duration {_DURATION}\n")
                for val in ts.value:
                    fw.write(f"{val:.16e}\n")
            print(f"  -> saved in {time.time()-t0:.1f}s")

    print("All detectors ready.")

In [ ]:
from __future__ import annotations

import os, sys, time
from functools import partial

import numpy as np
from scipy.interpolate import interp1d

# Use GPU if available on Colab, otherwise CPU
if "google.colab" not in sys.modules:
    os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

## Load the waveform model and monkey-patch SHARPy

We replace the original `IMRPhenomD` template in SHARPy with our `mlgw_bns_jax` model.

In [ ]:
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

# ---- Monkey-patch SHARPy's template -----------------------------------
import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses


def _template_mlgw_bns(params, frequency_array):
    """mlgw_bns_jax waveform, drop-in replacement for SHARPy's template."""
    mc, q = params[6], params[7]
    m1, m2 = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2 = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic = params[4]
    dist_mpc = jnp.exp(params[2])
    inclination = params[3]

    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor


_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
from sharpy.smc_functions import run_sharpy
import sharpy.PSDs

print("Model loaded — SHARPy template patched with mlgw_bns_jax.")

## Event parameters

In [ ]:
TRIGGER_TIME = 1187008882.43
SEGMENT_DURATION = 128.0        # paper-matching segment length
SAMPLING_RATE = 4096
F_LOWER = 23.0
F_UPPER = 2000.0
N_FREQ_POINTS = 4000            # reduced frequency grid size
DATA_START_GPS = 1187008114     # 1024s data file start
DATA_DURATION = 1024            # total data length (s)

# Fixed sky location: NGC 4993 (EM counterpart of GW170817)
FIXED_RA  = 3.44616     # rad
FIXED_DEC = -0.408084   # rad

DATA_DIR = "gw170817_data"
OUTDIR = "results_mlgw_bns_jax_pe_lowgrid"
LABEL = "GW170817_mlgw_bns_jax_lowgrid"
os.makedirs(OUTDIR, exist_ok=True)

print(f"Segment duration: {SEGMENT_DURATION}s  →  original df = {1/SEGMENT_DURATION:.4f} Hz")
print(f"Reduced grid: {N_FREQ_POINTS} points in [{F_LOWER}, {F_UPPER}] Hz")
print(f"  df_coarse = {(F_UPPER - F_LOWER) / (N_FREQ_POINTS - 1):.4f} Hz")
print(f"  reduction: ~{253000 / N_FREQ_POINTS:.0f}x vs full FFT grid")
print(f"Fixed sky: RA = {FIXED_RA:.5f} rad, Dec = {FIXED_DEC:.6f} rad  (NGC 4993)")

## Load cleaned data and build detector network

We use 1024 s of GWOSC strain data. For L1, the scatter-light glitch near
the merger was subtracted using a BayesWave-cleaned frame.

In [ ]:
# BayesWave-cleaned 1024s files (L1 BW-cleaned, H1/V1 raw GWOSC)
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

## Resample onto reduced frequency grid

The key trick: absorb the `exp(-2πi f (T-1))` phase from `project_waveform`
into the data **before** resampling.  This makes the data slowly-varying,
so interpolation onto a coarse grid is accurate.

After resampling, we monkey-patch `project_waveform` to remove the `(T-1)` term
that has already been absorbed.

In [ ]:
from scipy.interpolate import interp1d as _interp1d
import sharpy.GW_likelihood as _gw_mod
from sharpy.GW_likelihood import antenna_pattern_functions
from sharpy.utils import TimeDelayFromEarthCenter

# Access SHARPy's batched detector arrays
batched_det = gw_network.batched_detector

# Extract original frequency grid (same for all detectors after SHARPy build)
f_orig = np.array(batched_det.Frequency[0])
n_det = len(batched_det.latitude)

f_new = np.linspace(F_LOWER, F_UPPER, N_FREQ_POINTS)
df_new = f_new[1] - f_new[0]

print(f"Original FFT grid : {len(f_orig)} bins, df = {1/SEGMENT_DURATION:.4f} Hz")
print(f"New uniform grid  : {N_FREQ_POINTS} bins, df = {df_new:.4f} Hz  ({len(f_orig)/N_FREQ_POINTS:.0f}x reduction)\n")

print(f"Phase rotation: absorbing exp(+2pi i f x {SEGMENT_DURATION - 1:.0f}s) "
      f"into data before resampling")
print(f"  Nyquist time (coarse grid): {1/(2*df_new):.2f} s  "
      f"->  max |td+dtc| ~ 0.13 s is safe\n")

# Resample each detector
new_FrequencySeries_list = []
new_PSD_list = []

for i in range(n_det):
    f_det = np.array(batched_det.Frequency[i])
    sf_det = np.array(batched_det.FrequencySeries[i])
    psd_det = np.array(batched_det.PowerSpectralDensity[i])

    # Phase rotation: absorb (duration - 1) phase into data
    phase_corr = 2.0 * np.pi * f_det * (SEGMENT_DURATION - 1.0)
    sf_rotated = sf_det * np.exp(1j * phase_corr)

    # Interpolate rotated frequency series
    sf_new_real = _interp1d(f_det, sf_rotated.real, kind='cubic',
                            bounds_error=False, fill_value=0.0)(f_new)
    sf_new_imag = _interp1d(f_det, sf_rotated.imag, kind='cubic',
                            bounds_error=False, fill_value=0.0)(f_new)
    sf_new = sf_new_real + 1j * sf_new_imag

    # Interpolate PSD in log-space for positivity
    log_psd = np.log(np.where(psd_det > 0, psd_det, 1e-100))
    psd_new = np.exp(
        _interp1d(f_det, log_psd, kind='cubic',
                  bounds_error=False, fill_value=np.log(1e-100))(f_new)
    )

    new_FrequencySeries_list.append(sf_new)
    new_PSD_list.append(psd_new)

# Overwrite the batched detector arrays with low-res versions
batched_det = batched_det.replace(
    Frequency=jnp.stack([jnp.array(f_new, dtype=jnp.float64)] * n_det),
    FrequencySeries=jnp.stack([jnp.array(s, dtype=jnp.complex128) for s in new_FrequencySeries_list]),
    PowerSpectralDensity=jnp.stack([jnp.array(p, dtype=jnp.float64) for p in new_PSD_list]),
    sigmasq=jnp.stack([jnp.array(p, dtype=jnp.float64) for p in new_PSD_list]),
    TwoDeltaTOverN=jnp.stack([jnp.float64(2.0 * df_new)] * n_det),
)

# Update the network object
gw_network.batched_detector = batched_det


# ── CRITICAL: Monkey-patch project_waveform to remove (T-1) from timeshift ──
# The data has been pre-rotated by exp(+2πi f (D-1)), absorbing the (D-1) phase.
# SHARPy's original project_waveform uses:
#     timeshift = timedelay + params[8] + (detector_dictionary.T - 1)
# We must remove the (T-1) term to avoid doubling the phase:
#     timeshift = timedelay + params[8]

def project_waveform_lowgrid(params, detector_dictionary):
    """project_waveform for phase-rotated (low-grid) data.

    Identical to SHARPy's project_waveform, except the timeshift does NOT
    include (T - 1) — that phase has been absorbed into the data during
    the resampling step.
    """
    f = detector_dictionary.Frequency
    h_plus, h_cross = _gw_mod.template(params, f)

    fplus, fcross = antenna_pattern_functions(
        params,
        detector_dictionary.latitude, detector_dictionary.longitude,
        detector_dictionary.gamma, detector_dictionary.zeta,
        detector_dictionary.trigtime,
    )

    ra = params[0]
    dec = params[1]
    tc = detector_dictionary.trigtime + params[8]

    timedelay = TimeDelayFromEarthCenter(
        detector_dictionary.latitude, detector_dictionary.longitude,
        detector_dictionary.elevation, ra, dec, tc,
    )

    # Only td + delta_tc — the (T-1) phase is already in the rotated data
    timeshift = timedelay + params[8]
    shift = 2.0 * np.pi * f * timeshift

    h = (fplus * h_plus + fcross * h_cross) * (jnp.cos(shift) - 1j * jnp.sin(shift))
    return h


_gw_mod.project_waveform = project_waveform_lowgrid
print("Monkey-patched project_waveform for low-grid (removed T-1 phase).")

print(f"\nNetwork resampled to {N_FREQ_POINTS}-point uniform grid (trigger-frame).")
for i, det_name in enumerate(["H1", "L1", "V1"]):
    psd_vals = new_PSD_list[i]
    print(f"  {det_name}: {N_FREQ_POINTS} freq bins, "
          f"PSD range [{psd_vals.min():.2e}, {psd_vals.max():.2e}]")

## Define likelihood and priors

11 parameters sampled (RA and Dec fixed to NGC 4993), with priors matching the paper:

| Parameter | Range | Boundary |
|-----------|-------|----------|
| log(D_L/Mpc) | [0, ln 75] | reflective |
| iota | [0, π] | reflective |
| phi_c | [0, 2π] | periodic |
| psi | [0, π] | periodic |
| Mc | [1.18, 1.21] M_sun | reflective |
| q | [0.5, 1.0] | reflective |
| tc | [-0.1, 0.1] s | reflective |
| chi_1 | [-0.5, 0.5] | reflective |
| chi_2 | [-0.5, 0.5] | reflective |
| Lambda_1 | [5, 5000] | reflective |
| Lambda_2 | [5, 5000] | reflective |

In [ ]:
batched_detector = gw_network.batched_detector
log_likelihood_full = partial(log_likelihood_det, detector_list=batched_detector)


def log_likelihood_reduced(params_11):
    """Insert fixed RA/Dec and evaluate the full 13-param likelihood.

    params_11 layout:
        [0] logdist, [1] incl, [2] phic, [3] pol,
        [4] mc, [5] q, [6] tc, [7] chi1, [8] chi2,
        [9] lambda_1, [10] lambda_2
    """
    params_13 = jnp.concatenate([
        jnp.array([FIXED_RA, FIXED_DEC]),   # [0] ra, [1] dec  (fixed)
        params_11[:4],                       # [2] logdist, [3] incl, [4] phic, [5] pol
        params_11[4:9],                      # [6] mc, [7] q, [8] tc, [9] chi1, [10] chi2
        params_11[9:11],                     # [11] lambda_1, [12] lambda_2
    ])
    return log_likelihood_full(params_13)


# Prior bounds for the 11 sampled parameters
prior_bounds = jnp.array([
    [jnp.log(1.0),  jnp.log(75.0)],     # [0]  logdistance (1–75 Mpc)
    [0.0,           jnp.pi],             # [1]  inclination
    [0.0,           2 * jnp.pi],         # [2]  phic
    [0.0,           jnp.pi],             # [3]  pol
    [1.18,          1.21],               # [4]  mc  (chirp mass, M_sun)
    [0.5,           1.0],                # [5]  q   (mass ratio)
    [-0.1,          0.1],                # [6]  tc  (relative to trigger, s)
    [-0.5,          0.5],                # [7]  chi1
    [-0.5,          0.5],                # [8]  chi2
    [5.0,           5000.0],             # [9]  lambda_1
    [5.0,           5000.0],             # [10] lambda_2
])

# 1 = periodic, 0 = reflective
boundary_conditions = jnp.array([
    0,  # logdist  (reflective)
    0,  # incl     (reflective)
    1,  # phic     (periodic)
    1,  # pol      (periodic)
    0,  # mc       (reflective)
    0,  # q        (reflective)
    0,  # tc       (reflective)
    0,  # chi1     (reflective)
    0,  # chi2     (reflective)
    0,  # lambda_1 (reflective)
    0,  # lambda_2 (reflective)
])

parameter_names = [
    "logdistance", "theta_jn", "phiref", "pol",
    "mc", "q", "tc", "chi1", "chi2", "lambda_1", "lambda_2",
]


def prior(params):
    """Uniform prior (log-prior = 0 inside bounds)."""
    return 0.0


print(f"Fixed: RA = {FIXED_RA:.5f}, Dec = {FIXED_DEC:.6f}")
print(f"Sampling {len(parameter_names)} parameters: {parameter_names}")

In [ ]:
# Quick sanity check: likelihood at literature values
# GW170817 literature: mc~1.186, q~0.87, D~40 Mpc, iota~2.5 rad
# params_11: [logdist, incl, phic, pol, mc, q, tc, chi1, chi2, lambda_1, lambda_2]

# JIT-compile to avoid repeated tracing overhead
log_likelihood_jit = jax.jit(log_likelihood_reduced)

# Warm-up (single JIT compilation)
_warmup = log_likelihood_jit(jnp.array([
    jnp.log(40.0), jnp.pi/2, 0.0, 0.0,
    1.186, 0.87, 0.0, 0.0, 0.0,
    300.0, 300.0
]))
_warmup.block_until_ready()
print("JIT warm-up done.")

# Evaluate at a few test points
test_points = [
    ("Literature approx", [jnp.log(40.0), 2.5, 1.0, 0.5,
                           1.186, 0.87, 0.0, 0.0, 0.0, 300.0, 300.0]),
    ("Refined best-fit",  [jnp.log(56.0), 1.99, 1.0, 0.5,
                           1.1955, 0.54, 0.0, 0.0, 0.0, 300.0, 300.0]),
]
for label, p in test_points:
    logL = float(log_likelihood_jit(jnp.array(p)))
    print(f"  {label}: logL = {logL:.2f}")

# Compare to noise-only (zero signal)
logL_noise = float(log_likelihood_jit(jnp.array([
    jnp.log(1000.0), 0.0, 0.0, 0.0,
    1.186, 0.87, 0.0, 0.0, 0.0, 300.0, 300.0
])))
print(f"  Noise-only (D=1000 Mpc): logL = {logL_noise:.2f}")
print(f"  Delta logL (best vs noise): {float(log_likelihood_jit(jnp.array(test_points[1][1]))) - logL_noise:.2f}")

## Run the SMC sampler

With the reduced frequency grid, each likelihood evaluation is ~60x cheaper.
This allows running with many more particles on the same GPU.

In [ ]:
N_PARTICLES = 500
STEP_SIZE = 0.3
ALPHA = 0.95
SEED = 42

print(f"Starting SHARPy SMC with {N_PARTICLES} particles over {len(parameter_names)} parameters...")
start = time.time()

result_dict = run_sharpy(
    log_likelihood_reduced, prior,
    prior_bounds, boundary_conditions,
    ALPHA, N_PARTICLES, STEP_SIZE,
    jax.random.PRNGKey(SEED),
    folder=OUTDIR, label=LABEL,
)

dt = time.time() - start
samples = result_dict["posterior_samples"]
logZ, dlogZ = result_dict["logZ"], result_dict["dlogZ"]
print(f"\nDone in {dt:.1f} s — log Z = {logZ:.2f} ± {dlogZ:.2f}")

## Corner plot

In [ ]:
from corner import corner

fig = corner(
    np.array(samples), show_titles=True,
    labels=parameter_names, title_kwargs={"fontsize": 10},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved to {plot_path}")
fig

## Paper-style corner plot (Figure 9)

Compute derived parameters from the posterior samples and produce a corner plot
matching the paper layout: Mc, q, chi_eff, Lambda_tilde, D_L.

In [ ]:
from sharpy.utils import McQ2Masses

# Extract raw sampled parameters (11-param layout)
mc_samples   = np.array(samples[:, 4])   # chirp mass
q_samples    = np.array(samples[:, 5])   # mass ratio (m2/m1 <= 1)
chi1_samples = np.array(samples[:, 7])   # spin 1
chi2_samples = np.array(samples[:, 8])   # spin 2
lam1_samples = np.array(samples[:, 9])   # Lambda_1
lam2_samples = np.array(samples[:, 10])  # Lambda_2
logd_samples = np.array(samples[:, 0])   # log distance

# Compute component masses
m1_samples = np.zeros(len(mc_samples))
m2_samples = np.zeros(len(mc_samples))
for i in range(len(mc_samples)):
    m1_samples[i], m2_samples[i] = McQ2Masses(mc_samples[i], q_samples[i])

# chi_eff = (m1*chi1 + m2*chi2) / (m1 + m2)
chi_eff_samples = (m1_samples * chi1_samples + m2_samples * chi2_samples) / (m1_samples + m2_samples)

# Lambda_tilde (reduced tidal deformability)
M_samples = m1_samples + m2_samples
lambda_tilde_samples = (16.0 / 13.0) * (
    (m1_samples + 12.0 * m2_samples) * m1_samples**4 * lam1_samples
    + (m2_samples + 12.0 * m1_samples) * m2_samples**4 * lam2_samples
) / M_samples**5

# D_L in Mpc
dL_samples = np.exp(logd_samples)

# Build the 5-parameter array for the corner plot
paper_samples = np.column_stack([
    mc_samples,
    q_samples,
    chi_eff_samples,
    lambda_tilde_samples,
    dL_samples,
])

paper_labels = [
    r"$\mathcal{M}_c$ $[M_\odot]$",
    r"$q$",
    r"$\chi_{\rm eff}$",
    r"$\tilde{\Lambda}$",
    r"$D_L$ [Mpc]",
]

fig_paper = corner(
    paper_samples, show_titles=True,
    labels=paper_labels,
    title_kwargs={"fontsize": 12},
    quantiles=[0.05, 0.5, 0.95],
    levels=(0.5, 0.9),
    fill_contours=True,
    color="tab:orange",
)
plot_path_paper = os.path.join(OUTDIR, f"{LABEL}_corner_paper.png")
fig_paper.savefig(plot_path_paper, dpi=150)
print(f"Saved to {plot_path_paper}")
fig_paper